# Day 7 - Independent Evaluation on WikiANN English

This notebook evaluates the trained DeBERTa-v3-small checkpoint and the LLaMA zero-shot Template C pipeline on WikiANN English — an external benchmark dataset not used during project development.

Dataset: `wikiann/en` (test split)

Why this dataset:
- Completely external to the WikiNeural-derived training and test splits used earlier in the project.
- Pre-tokenised with integer NER tags; B-PER / I-PER are kept, all LOC / ORG tags are remapped to O.
- WikiANN has no EMAIL annotations — `email_f1` is reported as `null` throughout.

Outputs:
- `results/wikiann/data_stats.json`
- `results/wikiann/deberta_results.json`
- `results/wikiann/llama_results.json`
- `results/wikiann/eval_summary.json`

## 1. Setup

In [ ]:
import os
import pathlib
import sys

IN_KAGGLE = pathlib.Path('/kaggle/input').exists()
if IN_KAGGLE:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'seqeval',
        'datasets',
        'huggingface_hub',
        'llama-cpp-python',
        '--extra-index-url',
        'https://abetlen.github.io/llama-cpp-python/whl/cu121',
    ])

ROOT = pathlib.Path('/kaggle/working') if IN_KAGGLE else pathlib.Path.cwd().parent
PROJECT_SRC = pathlib.Path.cwd().parent / 'src'

if IN_KAGGLE:
    package_root = pathlib.Path('/kaggle/working/pii_masking')
    package_root.mkdir(parents=True, exist_ok=True)
    module_sources = {'__init__.py': '', 'day1_data.py': '"""Day 1 data loading, validation, and splitting utilities."""\n\nimport copy\nimport hashlib\nimport json\nimport logging\nimport statistics\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedShuffleSplit\n\nlogging.basicConfig(level=logging.INFO, format="%(message)s")\nlogger = logging.getLogger(__name__)\n\nVALID_TAGS = {"O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"}\nPUNCT_TOKENS = {".", "!", "?", ";"}\n\n\ndef validate_example(example: dict, idx: int) -> dict:\n    result = {"valid": True, "violations": [], "fixes_applied": []}\n    tokens = example.get("tokens", [])\n    ner_tags = example.get("ner_tags", [])\n\n    if len(tokens) != len(ner_tags):\n        result["valid"] = False\n        result["violations"].append(\n            f"Length mismatch: {len(tokens)} tokens vs {len(ner_tags)} tags"\n        )\n        return result\n\n    unknown = [t for t in ner_tags if t not in VALID_TAGS]\n    if unknown:\n        result["valid"] = False\n        result["violations"].append(f"Unknown tags: {sorted(set(unknown))}")\n        return result\n\n    if len(tokens) == 0 or any(t == "" for t in tokens):\n        result["valid"] = False\n        result["violations"].append("Empty token list or contains empty string tokens")\n        return result\n\n    fixed_tags = list(ner_tags)\n    for j, tag in enumerate(fixed_tags):\n        if not tag.startswith("I-"):\n            continue\n        entity_type = tag[2:]\n        prev_tag = fixed_tags[j - 1] if j > 0 else "O"\n        if prev_tag not in (f"B-{entity_type}", f"I-{entity_type}"):\n            fixed_tags[j] = f"B-{entity_type}"\n            fix_msg = f"Converted orphan I-{entity_type} at position {j} to B-{entity_type}"\n            result["fixes_applied"].append(fix_msg)\n            result["violations"].append(fix_msg)\n\n    if example.get("lang") != "en":\n        result["valid"] = False\n        result["violations"].append(f"Unexpected language: {example.get(\'lang\')!r}")\n\n    if result["fixes_applied"]:\n        fixed = copy.deepcopy(example)\n        fixed["ner_tags"] = fixed_tags\n        result["fixed_example"] = fixed\n\n    return result\n\n\ndef _count_spans(ner_tags: list[str], entity: str) -> int:\n    return sum(1 for t in ner_tags if t == f"B-{entity}")\n\n\ndef _density_bucket(n_per: int) -> str:\n    if n_per == 0:\n        return "zero_per"\n    if n_per == 1:\n        return "one_per"\n    return "multi_per"\n\n\ndef load_and_validate(json_path: str | Path) -> tuple[list[dict], dict]:\n    json_path = Path(json_path)\n    with json_path.open(encoding="utf-8") as f:\n        raw = json.load(f)\n\n    logger.info("Processing %d examples from %s...", len(raw), json_path.name)\n\n    cleaned: list[dict] = []\n    invalid_indices: list[int] = []\n    fix_log: list[dict] = []\n    bio_violations_fixed = 0\n    tag_counts: dict[str, int] = {}\n    lengths: list[int] = []\n\n    for idx, example in enumerate(raw):\n        val = validate_example(example, idx)\n        for tag in example.get("ner_tags", []):\n            tag_counts[tag] = tag_counts.get(tag, 0) + 1\n\n        if not val["valid"]:\n            invalid_indices.append(idx)\n            continue\n\n        effective = val.get("fixed_example", example)\n        for fix in val["fixes_applied"]:\n            bio_violations_fixed += 1\n            fix_log.append({"idx": idx, "fix_description": fix})\n\n        cleaned.append(effective)\n        lengths.append(len(effective["tokens"]))\n\n    per_counts = [_count_spans(e["ner_tags"], "PER") for e in cleaned]\n    zero_per = sum(1 for c in per_counts if c == 0)\n    one_per = sum(1 for c in per_counts if c == 1)\n    two_plus_per = sum(1 for c in per_counts if c >= 2)\n\n    email_count = sum(_count_spans(e["ner_tags"], "EMAIL") for e in cleaned)\n\n    length_stats: dict[str, float] = {}\n    if lengths:\n        sorted_l = sorted(lengths)\n        length_stats = {\n            "min": int(min(sorted_l)),\n            "max": int(max(sorted_l)),\n            "mean": round(statistics.mean(sorted_l), 2),\n            "median": float(statistics.median(sorted_l)),\n            "p95": float(np.percentile(sorted_l, 95)),\n        }\n\n    report = {\n        "source_file": str(json_path),\n        "total_examples": len(raw),\n        "valid_examples": len(cleaned),\n        "invalid_examples": len(invalid_indices),\n        "bio_violations_fixed": bio_violations_fixed,\n        "tag_distribution": tag_counts,\n        "length_stats": length_stats,\n        "per_span_counts": {\n            "0_per": zero_per,\n            "1_per": one_per,\n            "2plus_per": two_plus_per,\n        },\n        "email_span_count": email_count,\n        "invalid_indices": invalid_indices,\n        "fix_log": fix_log,\n    }\n\n    logger.info(\n        "  valid=%d  invalid=%d  bio_fixes=%d",\n        len(cleaned),\n        len(invalid_indices),\n        bio_violations_fixed,\n    )\n    return cleaned, report\n\n\ndef stratified_split(\n    examples: list[dict],\n    val_ratio: float = 0.15,\n    seed: int = 42,\n) -> tuple[list[dict], list[dict]]:\n    labels = [_density_bucket(_count_spans(e["ner_tags"], "PER")) for e in examples]\n    X = list(range(len(examples)))\n\n    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)\n    train_idx, val_idx = next(sss.split(X, labels))\n\n    train_examples = [examples[i] for i in train_idx]\n    val_examples = [examples[i] for i in val_idx]\n\n    from collections import Counter\n    train_buckets = Counter(_density_bucket(_count_spans(e["ner_tags"], "PER")) for e in train_examples)\n    val_buckets = Counter(_density_bucket(_count_spans(e["ner_tags"], "PER")) for e in val_examples)\n\n    logger.info(\n        "Split: train=%d %s | val=%d %s",\n        len(train_examples),\n        dict(train_buckets),\n        len(val_examples),\n        dict(val_buckets),\n    )\n    return train_examples, val_examples\n\n\ndef save_as_parquet(examples: list[dict], out_path: str | Path) -> None:\n    out_path = Path(out_path)\n    rows = []\n    for e in examples:\n        n_per = _count_spans(e["ner_tags"], "PER")\n        rows.append(\n            {\n                "lang": e.get("lang", "en"),\n                "sequence": e.get("sequence", ""),\n                "tokens": e["tokens"],\n                "ner_tags": e["ner_tags"],\n                "n_tokens": len(e["tokens"]),\n                "n_per_spans": n_per,\n                "n_email_spans": _count_spans(e["ner_tags"], "EMAIL"),\n                "entity_density_bucket": _density_bucket(n_per),\n            }\n        )\n    df = pd.DataFrame(rows)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_parquet(out_path, engine="pyarrow", index=False)\n    logger.info("Saved %d rows -> %s", len(df), out_path)\n\n\ndef compute_checksums(file_paths: list[str | Path]) -> dict[str, str]:\n    result: dict[str, str] = {}\n    for fp in file_paths:\n        fp = Path(fp)\n        h = hashlib.sha256()\n        with fp.open("rb") as f:\n            for chunk in iter(lambda: f.read(65536), b""):\n                h.update(chunk)\n        result[fp.name] = h.hexdigest()\n    return result\n', 'day4_llm_inference.py': '"""Day 4 LLM prompting and output parsing pipeline."""\n\nimport json\nimport os\nimport re\nimport string\nfrom typing import Optional\n\n_parse_failure_log: list[dict] = []\n\nSYSTEM_PROMPT = (\n    "You are a precise PII detection system. Your task is to identify person names and email addresses\\n"\n    "in text. Return ONLY a valid JSON object. No explanations. No preamble. No markdown. No commentary.\\n"\n    "Rules:\\n"\n    "- Extract full names as they appear (e.g., \\"John Smith\\", not \\"John\\" and \\"Smith\\" separately)\\n"\n    "- Do NOT extract honorifics like Dr., Mr., Mrs., Prof. as part of the name unless inseparable\\n"\n    "- Do NOT extract organizations, locations, or other entities - only person names and emails\\n"\n    "- If no names or emails are found, return empty lists"\n)\n\n\nclass LLMPIIPipeline:\n    def __init__(self, model_path: str, n_ctx: int = 2048, n_threads: int = 4, n_gpu_layers: int = 0):\n        from llama_cpp import Llama\n        self.llm = Llama(model_path=model_path, n_ctx=n_ctx, n_threads=n_threads,\n                         n_gpu_layers=n_gpu_layers, verbose=False)\n\n    def _build_prompt(self, sentence: str) -> str:\n        return (\n            "<|start_header_id|>system<|end_header_id|>\\n"\n            + SYSTEM_PROMPT\n            + "<|eot_id|><|start_header_id|>user<|end_header_id|>\\n"\n            "Identify all person names and email addresses in the following sentence.\\n"\n            "Return ONLY this JSON structure with no other text:\\n"\n            "{\\"names\\": [\\"...\\"], \\"emails\\": [\\"...\\"]}\\n"\n            "\\n"\n            "Sentence: " + sentence + "\\n"\n            "<|eot_id|><|start_header_id|>assistant<|end_header_id|>"\n        )\n\n    def _parse_response(\n        self, response: str, sentence: str = "", idx: int = -1\n    ) -> tuple[list[str], list[str], bool]:\n        text = response.strip()\n\n        if text.startswith("assistant"):\n            text = text[len("assistant"):].lstrip("\\n").strip()\n\n        fence_json = re.search(r"```json\\s*([\\s\\S]*?)```", text)\n        if fence_json:\n            text = fence_json.group(1).strip()\n        else:\n            fence_plain = re.search(r"```\\s*([\\s\\S]*?)```", text)\n            if fence_plain:\n                text = fence_plain.group(1).strip()\n\n        parsed = None\n        try:\n            parsed = json.loads(text)\n        except json.JSONDecodeError:\n            pass\n\n        if parsed is None:\n            first = text.find("{")\n            last = text.rfind("}")\n            if first != -1 and last != -1 and last > first:\n                try:\n                    parsed = json.loads(text[first : last + 1])\n                except json.JSONDecodeError:\n                    pass\n\n        if parsed is None:\n            first = text.find("{")\n            if first != -1:\n                fragment = text[first:]\n                try:\n                    names_match = re.search(r\'"names"\\s*:\\s*\\[([^\\]]*)\', fragment)\n                    emails_match = re.search(r\'"emails"\\s*:\\s*\\[([^\\]]*)\', fragment)\n                    recovered_names = re.findall(r\'"([^"]+)"\', names_match.group(1)) if names_match else []\n                    recovered_emails = re.findall(r\'"([^"]+)"\', emails_match.group(1)) if emails_match else []\n                    parsed = {"names": recovered_names, "emails": recovered_emails}\n                except Exception:\n                    pass\n\n        if parsed is None:\n            _parse_failure_log.append(\n                {"idx": idx, "sequence": sentence, "raw_response": response}\n            )\n            return [], [], False\n\n        names = [n for n in parsed.get("names", []) if isinstance(n, str) and n.strip()]\n        emails = [e for e in parsed.get("emails", []) if isinstance(e, str) and e.strip()]\n\n        sentence_lower = sentence.lower()\n        names = [n for n in names if n.strip().lower() in sentence_lower]\n        emails = [e for e in emails if e.strip().lower() in sentence_lower]\n\n        return names, emails, True\n\n    def _align_to_iob2(\n        self, tokens: list[str], names: list[str], emails: list[str]\n    ) -> list[str]:\n        emails_lower = [e.strip().lower() for e in emails]\n        names = [n for n in names if not any(n.strip().lower() in e for e in emails_lower)]\n\n        tags = ["O"] * len(tokens)\n        tokens_lower = [t.lower() for t in tokens]\n\n        def _strip_punct(s: str) -> str:\n            return s.strip(string.punctuation)\n\n        def _find_and_tag(entity: str, b_tag: str, i_tag: str) -> None:\n            if "@" in entity:\n                ent_lower = entity.strip().lower()\n                for i, tok in enumerate(tokens_lower):\n                    if _strip_punct(tok) == _strip_punct(ent_lower) and tags[i] == "O":\n                        tags[i] = b_tag\n                        return\n                parts = ent_lower.replace("@", " @ ").split()\n                if len(parts) >= 2:\n                    for start in range(len(tokens) - len(parts) + 1):\n                        window = [_strip_punct(tokens_lower[start + k]) for k in range(len(parts))]\n                        if window == [_strip_punct(p) for p in parts]:\n                            if all(tags[start + k] == "O" for k in range(len(parts))):\n                                tags[start] = b_tag\n                                for k in range(1, len(parts)):\n                                    tags[start + k] = i_tag\n                                return\n\n            ent_words = entity.strip().split()\n            if not ent_words:\n                return\n            ent_words_lower = [w.lower() for w in ent_words]\n            span_len = len(ent_words_lower)\n\n            for start in range(len(tokens) - span_len + 1):\n                window = [_strip_punct(tokens_lower[start + k]) for k in range(span_len)]\n                if window == [_strip_punct(w) for w in ent_words_lower]:\n                    if all(tags[start + k] == "O" for k in range(span_len)):\n                        tags[start] = b_tag\n                        for k in range(1, span_len):\n                            tags[start + k] = i_tag\n                        return\n\n        for email in emails:\n            _find_and_tag(email, "B-EMAIL", "I-EMAIL")\n        for name in names:\n            _find_and_tag(name, "B-PER", "I-PER")\n\n        return tags\n\n    def predict_sentence(self, tokens: list[str], sequence: str) -> dict:\n        prompt = self._build_prompt(sequence)\n        response = self.llm(\n            prompt,\n            max_tokens=256,\n            temperature=0.0,\n            stop=["<|eot_id|>", "<|end_of_text|>"],\n        )\n        raw_text = response["choices"][0]["text"]\n        names, emails, parse_ok = self._parse_response(raw_text, sentence=sequence)\n        predicted_tags = self._align_to_iob2(tokens, names, emails)\n\n        return {\n            "tokens": tokens,\n            "sequence": sequence,\n            "raw_response": raw_text,\n            "parsed_names": names,\n            "parsed_emails": emails,\n            "predicted_tags": predicted_tags,\n            "parse_ok": parse_ok,\n        }\n\n    def predict_batch(\n        self,\n        records: list[dict],\n        cache_path: str,\n        checkpoint_every: int = 50,\n    ) -> list[dict]:\n        processed_indices: set[int] = set()\n        cached_results: list[dict] = []\n        if os.path.exists(cache_path):\n            with open(cache_path, "r", encoding="utf-8") as f:\n                for line in f:\n                    line = line.strip()\n                    if not line:\n                        continue\n                    try:\n                        obj = json.loads(line)\n                        cached_results.append(obj)\n                        if "idx" in obj:\n                            processed_indices.add(obj["idx"])\n                    except json.JSONDecodeError:\n                        pass\n            print(f"Resuming: {len(cached_results)} records already in cache.")\n\n        results: list[dict] = list(cached_results)\n        total = len(records)\n        parse_failures = sum(1 for r in cached_results if not r.get("parse_ok", True))\n        newly_processed = 0\n\n        def _write_cache(path: str, data: list[dict]) -> None:\n            tmp = path + ".tmp"\n            with open(tmp, "w", encoding="utf-8") as f:\n                for obj in data:\n                    f.write(json.dumps(obj) + "\\n")\n            os.replace(tmp, path)\n\n        for idx, record in enumerate(records):\n            if idx in processed_indices:\n                continue\n\n            result = self.predict_sentence(record["tokens"], record["sequence"])\n            result["idx"] = idx\n            if "ner_tags" in record:\n                result["ner_tags"] = record["ner_tags"]\n\n            results.append(result)\n            if not result["parse_ok"]:\n                parse_failures += 1\n            newly_processed += 1\n\n            if newly_processed % 100 == 0:\n                print(\n                    f"Processed {idx + 1}/{total} sentences "\n                    f"(parse failures so far: {parse_failures})"\n                )\n\n            if newly_processed % checkpoint_every == 0:\n                _write_cache(cache_path, results)\n\n        _write_cache(cache_path, results)\n        print(\n            f"Done. Processed {total} sentences total "\n            f"(parse failures: {parse_failures})"\n        )\n        return results\n', 'eval_independent.py': '"""Independent Hugging Face dataset evaluation — WikiANN English test split."""\n\nfrom __future__ import annotations\n\nimport json\nimport logging\nimport pathlib\nimport re\nfrom dataclasses import dataclass\nfrom typing import Any\n\nimport numpy as np\nfrom seqeval.metrics import classification_report\nfrom seqeval.scheme import IOB2\n\nfrom pii_masking.day1_data import validate_example\nfrom pii_masking.day4_llm_inference import LLMPIIPipeline, _parse_failure_log\n\nlogger = logging.getLogger(__name__)\n\nLABEL2ID = {"O": 0, "B-PER": 1, "I-PER": 2, "B-EMAIL": 3, "I-EMAIL": 4}\nID2LABEL = {v: k for k, v in LABEL2ID.items()}\n\nNAME_LABELS = {\n    "FIRSTNAME",\n    "GIVENNAME",\n    "GIVENNAME1",\n    "GIVENNAME2",\n    "LASTNAME",\n    "LASTNAME1",\n    "LASTNAME2",\n    "MIDDLENAME",\n    "NAME",\n    "FULLNAME",\n    "SURNAME",\n    "USERNAME",\n}\nEMAIL_LABELS = {"EMAIL"}\nTOKEN_RE = re.compile(r"\\S+")\n\n\n@dataclass(frozen=True)\nclass EvalPaths:\n    results_dir: pathlib.Path\n    stats_path: pathlib.Path\n    deberta_path: pathlib.Path\n    llama_path: pathlib.Path\n    summary_path: pathlib.Path\n    llama_cache_path: pathlib.Path\n\n\ndef wikiann_paths(results_dir: str | pathlib.Path = "results/wikiann") -> EvalPaths:\n    root = pathlib.Path(results_dir)\n    return EvalPaths(\n        results_dir=root,\n        stats_path=root / "data_stats.json",\n        deberta_path=root / "deberta_results.json",\n        llama_path=root / "llama_results.json",\n        summary_path=root / "eval_summary.json",\n        llama_cache_path=root / "llama_outputs.jsonl",\n    )\n\n\ndef day7_paths(results_dir: str | pathlib.Path = "results/day7") -> EvalPaths:\n    root = pathlib.Path(results_dir)\n    return EvalPaths(\n        results_dir=root,\n        stats_path=root / "independent_data_stats.json",\n        deberta_path=root / "independent_deberta.json",\n        llama_path=root / "independent_llama.json",\n        summary_path=root / "independent_eval_summary.json",\n        llama_cache_path=root / "independent_llama_outputs.jsonl",\n    )\n\n\ndef _tokenize_with_offsets(text: str) -> tuple[list[str], list[tuple[int, int]]]:\n    tokens: list[str] = []\n    offsets: list[tuple[int, int]] = []\n    for match in TOKEN_RE.finditer(text):\n        tokens.append(match.group(0))\n        offsets.append((match.start(), match.end()))\n    return tokens, offsets\n\n\ndef _normalise_mask(mask: Any) -> list[dict[str, Any]]:\n    if isinstance(mask, str):\n        try:\n            mask = json.loads(mask)\n        except json.JSONDecodeError:\n            return []\n    return mask if isinstance(mask, list) else []\n\n\ndef _map_privacy_label(label: str) -> str | None:\n    normalized = str(label).upper().replace("-", "_")\n    if normalized in EMAIL_LABELS:\n        return "EMAIL"\n    if normalized in NAME_LABELS:\n        return "PER"\n    return None\n\n\ndef ai4privacy_row_to_example(row: dict[str, Any], idx: int) -> dict[str, Any] | None:\n    """Convert one ai4privacy row into project BIO tags for PER and EMAIL only."""\n    text = str(row.get("source_text") or "")\n    if not text.strip():\n        return None\n\n    tokens, offsets = _tokenize_with_offsets(text)\n    if not tokens:\n        return None\n\n    tags = ["O"] * len(tokens)\n    masks = _normalise_mask(row.get("privacy_mask", []))\n    mapped_spans: list[tuple[int, int, str]] = []\n    for item in masks:\n        if not isinstance(item, dict):\n            continue\n        entity = _map_privacy_label(str(item.get("label", "")))\n        if entity is None:\n            continue\n        try:\n            start = int(item["start"])\n            end = int(item["end"])\n        except (KeyError, TypeError, ValueError):\n            continue\n        mapped_spans.append((start, end, entity))\n\n    merged_spans = merge_adjacent_name_spans(text, mapped_spans)\n    for start, end, entity in merged_spans:\n        covered = [\n            tok_idx\n            for tok_idx, (tok_start, tok_end) in enumerate(offsets)\n            if tok_start < end and tok_end > start\n        ]\n        if not covered:\n            continue\n\n        for pos, tok_idx in enumerate(covered):\n            if tags[tok_idx] != "O":\n                continue\n            tags[tok_idx] = f"B-{entity}" if pos == 0 else f"I-{entity}"\n\n    return {\n        "idx": idx,\n        "source_dataset": "ai4privacy/pii-masking-300k",\n        "lang": row.get("language", "unknown"),\n        "sequence": text,\n        "tokens": tokens,\n        "ner_tags": tags,\n    }\n\n\ndef merge_adjacent_name_spans(\n    text: str,\n    spans: list[tuple[int, int, str]],\n) -> list[tuple[int, int, str]]:\n    """Merge adjacent PER spans separated only by whitespace or light punctuation."""\n    if not spans:\n        return []\n\n    sorted_spans = sorted(spans, key=lambda span: (span[0], span[1]))\n    merged: list[tuple[int, int, str]] = []\n    for start, end, entity in sorted_spans:\n        if not merged:\n            merged.append((start, end, entity))\n            continue\n\n        prev_start, prev_end, prev_entity = merged[-1]\n        gap = text[prev_end:start]\n        can_merge_name = (\n            prev_entity == "PER"\n            and entity == "PER"\n            and re.fullmatch(r"[\\s.\'-]{0,4}", gap or "") is not None\n        )\n        if can_merge_name:\n            merged[-1] = (prev_start, end, "PER")\n        else:\n            merged.append((start, end, entity))\n    return merged\n\n\ndef load_ai4privacy_examples(\n    dataset_name: str = "ai4privacy/pii-masking-300k",\n    split: str = "train",\n    max_records: int = 1000,\n    language: str = "English",\n    min_entity_spans: int = 1,\n) -> list[dict[str, Any]]:\n    """Load an external Hugging Face PII dataset and map it to project BIO labels."""\n    from datasets import load_dataset\n\n    dataset = load_dataset(dataset_name, split=split, streaming=True)\n    examples: list[dict[str, Any]] = []\n    for idx, row in enumerate(dataset):\n        row_language = str(row.get("language", ""))\n        if language and row_language.lower() != language.lower():\n            continue\n        example = ai4privacy_row_to_example(row, idx)\n        if example is None:\n            continue\n        span_count = count_spans(example["ner_tags"], "PER") + count_spans(\n            example["ner_tags"], "EMAIL"\n        )\n        if span_count < min_entity_spans:\n            continue\n        examples.append(example)\n        if len(examples) >= max_records:\n            break\n\n    if not examples:\n        raise ValueError(\n            f"No usable examples loaded from {dataset_name} split={split!r} language={language!r}"\n        )\n    return examples\n\n\ndef wikiann_row_to_example(\n    row: dict[str, Any],\n    idx: int,\n    label_names: list[str],\n) -> dict[str, Any] | None:\n    """Convert one WikiANN row to project BIO tags keeping only PER; drop LOC/ORG."""\n    tokens: list[str] = list(row.get("tokens") or [])\n    raw_tags: list[int] = list(row.get("ner_tags") or [])\n    if not tokens:\n        return None\n\n    _PER_KEEP = {"B-PER", "I-PER"}\n    tags: list[str] = []\n    for tag_id in raw_tags:\n        decoded = label_names[tag_id] if tag_id < len(label_names) else "O"\n        tags.append(decoded if decoded in _PER_KEEP else "O")\n\n    return {\n        "idx": idx,\n        "source_dataset": "wikiann/en",\n        "lang": "en",\n        "sequence": " ".join(tokens),\n        "tokens": tokens,\n        "ner_tags": tags,\n    }\n\n\ndef load_wikiann_examples(\n    split: str = "test",\n    min_entity_spans: int = 1,\n) -> list[dict[str, Any]]:\n    """Load WikiANN English split and map to project BIO labels (PER only)."""\n    from datasets import load_dataset\n\n    ds = load_dataset("wikiann", "en", split=split)\n    label_names: list[str] = ds.features["ner_tags"].feature.names\n\n    examples: list[dict[str, Any]] = []\n    for idx, row in enumerate(ds):\n        example = wikiann_row_to_example(row, idx, label_names)\n        if example is None:\n            continue\n        if count_spans(example["ner_tags"], "PER") < min_entity_spans:\n            continue\n        examples.append(example)\n\n    if not examples:\n        raise ValueError(f"No usable examples loaded from wikiann/en split={split!r}")\n    return examples\n\n\ndef count_spans(tags: list[str], entity: str) -> int:\n    return sum(1 for tag in tags if tag == f"B-{entity}")\n\n\ndef extract_spans(tags: list[str]) -> list[tuple[int, int, str]]:\n    spans: list[tuple[int, int, str]] = []\n    i = 0\n    while i < len(tags):\n        tag = tags[i]\n        if tag.startswith("B-"):\n            entity = tag[2:]\n            j = i + 1\n            while j < len(tags) and tags[j] == f"I-{entity}":\n                j += 1\n            spans.append((i, j, entity))\n            i = j\n        else:\n            i += 1\n    return spans\n\n\ndef inspect_examples(\n    examples: list[dict[str, Any]],\n    dataset_name: str | None = None,\n) -> dict[str, Any]:\n    valid = 0\n    invalid = 0\n    fixes = 0\n    token_count = 0\n    per_spans = 0\n    email_spans = 0\n    violations: list[dict[str, Any]] = []\n\n    for idx, example in enumerate(examples):\n        check = validate_example(example, idx)\n        if check["valid"]:\n            valid += 1\n        else:\n            invalid += 1\n            violations.append({"idx": idx, "violations": check["violations"]})\n        fixes += len(check.get("fixes_applied", []))\n        token_count += len(example["tokens"])\n        per_spans += count_spans(example["ner_tags"], "PER")\n        email_spans += count_spans(example["ner_tags"], "EMAIL")\n\n    inferred_name = dataset_name or (\n        examples[0].get("source_dataset", "unknown") if examples else "unknown"\n    )\n    return {\n        "dataset": inferred_name,\n        "n_sentences": len(examples),\n        "n_valid_sentences": valid,\n        "n_invalid_sentences": invalid,\n        "bio_fixes_applied": fixes,\n        "n_tokens": token_count,\n        "per_span_count": per_spans,\n        "email_span_count": email_spans,\n        "email_metrics_note": None\n        if email_spans\n        else "Dataset contains no EMAIL spans; EMAIL metrics are reported as null.",\n        "violations": violations[:50],\n    }\n\n\ndef save_json(payload: dict[str, Any], path: str | pathlib.Path) -> None:\n    path = pathlib.Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n\n\ndef predict_deberta(\n    examples: list[dict[str, Any]],\n    model_dir: str | pathlib.Path,\n    batch_size: int = 16,\n    max_length: int = 256,\n) -> list[dict[str, Any]]:\n    import torch\n    from transformers import AutoModelForTokenClassification, AutoTokenizer\n\n    model_dir = pathlib.Path(model_dir)\n    tokenizer = AutoTokenizer.from_pretrained(str(model_dir))\n    model = AutoModelForTokenClassification.from_pretrained(str(model_dir))\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    model.to(device)\n    model.eval()\n\n    id2label = {int(k): v for k, v in model.config.id2label.items()}\n    results: list[dict[str, Any]] = []\n\n    for batch_start in range(0, len(examples), batch_size):\n        batch = examples[batch_start : batch_start + batch_size]\n        batch_tokens = [record["tokens"] for record in batch]\n        encoding = tokenizer(\n            batch_tokens,\n            is_split_into_words=True,\n            padding=True,\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        )\n        model_inputs = {\n            "input_ids": encoding["input_ids"].to(device),\n            "attention_mask": encoding["attention_mask"].to(device),\n        }\n        if "token_type_ids" in encoding:\n            model_inputs["token_type_ids"] = encoding["token_type_ids"].to(device)\n\n        with torch.no_grad():\n            pred_ids = model(**model_inputs).logits.argmax(dim=-1).cpu().numpy()\n\n        for batch_idx, record in enumerate(batch):\n            word_ids = encoding.word_ids(batch_index=batch_idx)\n            word_pred: dict[int, str] = {}\n            for token_idx, word_id in enumerate(word_ids):\n                if word_id is None or word_id in word_pred:\n                    continue\n                word_pred[word_id] = id2label.get(int(pred_ids[batch_idx][token_idx]), "O")\n\n            pred_tags = [word_pred.get(i, "O") for i in range(len(record["tokens"]))]\n            results.append(\n                {\n                    "idx": record.get("idx", batch_start + batch_idx),\n                    "tokens": record["tokens"],\n                    "sequence": record["sequence"],\n                    "gold_tags": record["ner_tags"],\n                    "pred_tags": pred_tags,\n                }\n            )\n\n    return results\n\n\ndef predict_llama(\n    examples: list[dict[str, Any]],\n    model_path: str | pathlib.Path,\n    cache_path: str | pathlib.Path,\n    n_threads: int = 4,\n    n_gpu_layers: int = 0,\n    checkpoint_every: int = 50,\n) -> list[dict[str, Any]]:\n    _parse_failure_log.clear()\n    pipeline = LLMPIIPipeline(\n        model_path=str(model_path),\n        n_threads=n_threads,\n        n_gpu_layers=n_gpu_layers,\n    )\n    records = [\n        {\n            "tokens": example["tokens"],\n            "sequence": example["sequence"],\n            "ner_tags": example["ner_tags"],\n        }\n        for example in examples\n    ]\n    raw = pipeline.predict_batch(\n        records,\n        cache_path=str(cache_path),\n        checkpoint_every=checkpoint_every,\n    )\n    raw.sort(key=lambda row: row.get("idx", 0))\n    return [\n        {\n            "idx": row.get("idx", i),\n            "tokens": row["tokens"],\n            "sequence": row["sequence"],\n            "gold_tags": row.get("ner_tags", []),\n            "pred_tags": row.get("predicted_tags", []),\n            "parse_ok": bool(row.get("parse_ok", False)),\n        }\n        for i, row in enumerate(raw)\n    ]\n\n\ndef compute_metrics(records: list[dict[str, Any]], email_present: bool) -> dict[str, Any]:\n    gold_seqs: list[list[str]] = []\n    pred_seqs: list[list[str]] = []\n    for record in records:\n        gold = list(record.get("gold_tags", []))\n        pred = (list(record.get("pred_tags", [])) + ["O"] * len(gold))[: len(gold)]\n        gold_seqs.append(gold)\n        pred_seqs.append(pred)\n\n    report = classification_report(\n        gold_seqs,\n        pred_seqs,\n        mode="strict",\n        scheme=IOB2,\n        output_dict=True,\n        zero_division=0,\n    )\n    token_rates = compute_token_rates(records)\n    leak = compute_span_leak_rate(records)\n\n    return {\n        "per_f1": float(report.get("PER", {}).get("f1-score", 0.0)),\n        "email_f1": (\n            float(report.get("EMAIL", {}).get("f1-score", 0.0)) if email_present else None\n        ),\n        "overall_f1": float(report.get("macro avg", {}).get("f1-score", 0.0)),\n        "fpr": token_rates["fpr"],\n        "fnr": token_rates["fnr"],\n        "redaction_leak_rate": leak["redaction_leak_rate"],\n        "leaked_span_count": leak["leaked_span_count"],\n        "total_pii_span_count": leak["total_pii_span_count"],\n    }\n\n\ndef compute_token_rates(records: list[dict[str, Any]]) -> dict[str, float]:\n    fp = fn = tp = tn = 0\n    for record in records:\n        gold = list(record.get("gold_tags", []))\n        pred = (list(record.get("pred_tags", [])) + ["O"] * len(gold))[: len(gold)]\n        for gold_tag, pred_tag in zip(gold, pred):\n            gold_pii = gold_tag != "O"\n            pred_pii = pred_tag != "O"\n            if gold_pii and pred_pii:\n                tp += 1\n            elif not gold_pii and pred_pii:\n                fp += 1\n            elif gold_pii and not pred_pii:\n                fn += 1\n            else:\n                tn += 1\n    return {\n        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,\n        "fnr": fn / (fn + tp) if (fn + tp) else 0.0,\n    }\n\n\ndef compute_span_leak_rate(records: list[dict[str, Any]]) -> dict[str, Any]:\n    leaked = 0\n    total = 0\n    for record in records:\n        gold = list(record.get("gold_tags", []))\n        pred = (list(record.get("pred_tags", [])) + ["O"] * len(gold))[: len(gold)]\n        for start, end, _entity in extract_spans(gold):\n            total += 1\n            if any(pred[pos] == "O" for pos in range(start, end)):\n                leaked += 1\n    return {\n        "redaction_leak_rate": leaked / total if total else 0.0,\n        "leaked_span_count": leaked,\n        "total_pii_span_count": total,\n    }\n\n\ndef choose_best_deberta_model(models_dir: str | pathlib.Path = "models") -> pathlib.Path:\n    """Pick the best local DeBERTa seed by result JSON, falling back to seed 42."""\n    models_dir = pathlib.Path(models_dir)\n    candidates: list[tuple[float, pathlib.Path]] = []\n    for result_path in models_dir.glob("deberta_seed*/deberta_seed*_result.json"):\n        try:\n            payload = json.loads(result_path.read_text(encoding="utf-8"))\n        except json.JSONDecodeError:\n            continue\n        metrics = payload.get("test_metrics") or payload.get("val_metrics") or payload\n        score = (\n            metrics.get("eval_overall_f1")\n            or metrics.get("overall_f1")\n            or metrics.get("f1")\n            or 0.0\n        )\n        model_dir = result_path.parent / "best_model"\n        if model_dir.exists():\n            candidates.append((float(score), model_dir))\n\n    if candidates:\n        return max(candidates, key=lambda item: item[0])[1]\n    return models_dir / "deberta_seed42" / "best_model"\n\n\ndef build_summary(\n    dataset_name: str,\n    stats: dict[str, Any],\n    deberta_metrics: dict[str, Any],\n    llama_metrics: dict[str, Any],\n) -> dict[str, Any]:\n    return {\n        "dataset": dataset_name,\n        "n_sentences": stats["n_sentences"],\n        "deberta": {\n            "per_f1": deberta_metrics["per_f1"],\n            "email_f1": deberta_metrics["email_f1"],\n            "overall_f1": deberta_metrics["overall_f1"],\n            "fpr": deberta_metrics["fpr"],\n            "fnr": deberta_metrics["fnr"],\n            "redaction_leak_rate": deberta_metrics["redaction_leak_rate"],\n        },\n        "llama": {\n            "per_f1": llama_metrics["per_f1"],\n            "email_f1": llama_metrics["email_f1"],\n            "overall_f1": llama_metrics["overall_f1"],\n            "fpr": llama_metrics["fpr"],\n            "fnr": llama_metrics["fnr"],\n            "redaction_leak_rate": llama_metrics["redaction_leak_rate"],\n            "parse_failure_rate": llama_metrics.get("parse_failure_rate", 0.0),\n        },\n    }\n'}
    for filename, content in module_sources.items():
        (package_root / filename).write_text(content, encoding='utf-8')
    if '/kaggle/working' not in sys.path:
        sys.path.insert(0, '/kaggle/working')
else:
    if str(PROJECT_SRC) not in sys.path:
        sys.path.insert(0, str(PROJECT_SRC))

from pii_masking.eval_independent import (
    build_summary,
    choose_best_deberta_model,
    compute_metrics,
    inspect_examples,
    load_wikiann_examples,
    predict_deberta,
    predict_llama,
    save_json,
    wikiann_paths,
)

print('IN_KAGGLE:', IN_KAGGLE)
print('ROOT:', ROOT)

## 2. Configuration

In [ ]:
DATASET_SPLIT = 'test'

RESULTS_DIR = pathlib.Path('/kaggle/working/results/wikiann') if IN_KAGGLE else pathlib.Path('../results/wikiann')
paths = wikiann_paths(RESULTS_DIR)
paths.results_dir.mkdir(parents=True, exist_ok=True)

LOCAL_DEBERTA = pathlib.Path('../models/deberta_seed7/best_model')
LOCAL_LLAMA = pathlib.Path('../models/llama/Llama-3.2-1B-Instruct-Q4_K_M.gguf')

print('Dataset: wikiann/en')
print('Split:', DATASET_SPLIT)
print('Results:', paths.results_dir)

## 3. Kaggle Model Path Discovery

In [ ]:
def find_first(root, pattern):
    root = pathlib.Path(root)
    if not root.exists():
        return None
    matches = sorted(root.rglob(pattern))
    return matches[0] if matches else None

if IN_KAGGLE:
    DEBERTA_MODEL_DIR = find_first('/kaggle/input', 'deberta_seed7/best_model')
    if DEBERTA_MODEL_DIR is None:
        DEBERTA_MODEL_DIR = find_first('/kaggle/input', 'deberta_seed42/best_model')
    LLAMA_MODEL_PATH = find_first('/kaggle/input', 'Llama-3.2-1B-Instruct-Q4_K_M.gguf')
    if LLAMA_MODEL_PATH is None:
        from huggingface_hub import hf_hub_download
        LLAMA_MODEL_PATH = pathlib.Path(hf_hub_download(
            repo_id='bartowski/Llama-3.2-1B-Instruct-GGUF',
            filename='Llama-3.2-1B-Instruct-Q4_K_M.gguf',
            local_dir='/kaggle/working/models/llama',
        ))
else:
    DEBERTA_MODEL_DIR = choose_best_deberta_model('../models') if pathlib.Path('../models').exists() else LOCAL_DEBERTA
    LLAMA_MODEL_PATH = LOCAL_LLAMA

print('DeBERTa model dir:', DEBERTA_MODEL_DIR)
print('LLaMA GGUF:', LLAMA_MODEL_PATH)

if DEBERTA_MODEL_DIR is None:
    raise FileNotFoundError('Could not find a DeBERTa best_model directory. Add the Day 3 training output as a Kaggle input.')
if LLAMA_MODEL_PATH is None:
    raise FileNotFoundError('Could not find Llama-3.2-1B-Instruct-Q4_K_M.gguf. Add the Day 4 LLaMA model/input or download it before inference.')

## 4. Load and Inspect Independent Data

In [ ]:
examples = load_wikiann_examples(split=DATASET_SPLIT)
stats = inspect_examples(examples)
stats.update({
    'dataset': 'wikiann/en',
    'split': DATASET_SPLIT,
})
save_json(stats, paths.stats_path)

print('Sentences:', stats['n_sentences'])
print('Tokens:', stats['n_tokens'])
print('PER spans:', stats['per_span_count'])
print('EMAIL spans:', stats['email_span_count'])
print('Saved:', paths.stats_path)
if stats['email_metrics_note']:
    print(stats['email_metrics_note'])

## 5. Approach A - DeBERTa Inference

In [ ]:
email_present = stats['email_span_count'] > 0
deberta_records = predict_deberta(examples, model_dir=DEBERTA_MODEL_DIR, batch_size=16)
deberta_metrics = compute_metrics(deberta_records, email_present=email_present)
deberta_metrics['model_dir'] = str(DEBERTA_MODEL_DIR)
save_json(deberta_metrics, paths.deberta_path)
deberta_metrics

## 6. Approach B - LLaMA Template C Inference

In [ ]:
llama_records = predict_llama(
    examples,
    model_path=LLAMA_MODEL_PATH,
    cache_path=paths.llama_cache_path,
    n_threads=4,
    n_gpu_layers=-1 if IN_KAGGLE else 0,
    checkpoint_every=50,
)
llama_metrics = compute_metrics(llama_records, email_present=email_present)
parse_failures = sum(1 for row in llama_records if not row.get('parse_ok', False))
llama_metrics['parse_failure_rate'] = parse_failures / len(llama_records)
llama_metrics['parse_failure_count'] = parse_failures
llama_metrics['model_path'] = str(LLAMA_MODEL_PATH)
save_json(llama_metrics, paths.llama_path)
llama_metrics

## 7. Consolidated Results

In [ ]:
summary = build_summary('wikiann/en', stats, deberta_metrics, llama_metrics)
save_json(summary, paths.summary_path)
summary

In [ ]:
import pandas as pd

rows = []
for model_name in ['deberta', 'llama']:
    row = {'model': model_name}
    row.update(summary[model_name])
    rows.append(row)

pd.DataFrame(rows)